# Novidades do MissingPatterns.jl 0.8 — um exemplo em português

A versão 0.7 do [MissingPatterns.jl](https://github.com/dantebertuzzi/MissingPatterns.jl) trouxe dois
testes — o de Little para MCAR e `missingpredictors` — e o
[notebook dela](https://colab.research.google.com/github/dantebertuzzi/MissingPatterns.jl/blob/main/notebooks/novidades-0.7.ipynb) terminou discutindo um falso positivo corrigido à mão. A 0.8
torna esses testes mais fáceis de ler direito e acrescenta uma visualização:

- **p-valores ajustados** em `missingpredictors` — a nova coluna `padj` corrige para os vários
  testes feitos de uma vez (Holm, por padrão, ou Benjamini–Hochberg);
- **testes impressos** — `missingmcar(io, tbl)` e `missingpredictors(io, tbl, col)` mostram o
  resultado com uma leitura, e o teste MCAR passa a dizer quais colunas deixou de fora;
- **`missinggaps`** — se a ausência de cada coluna está em buracos isolados ou em blocos contínuos.

A 0.8 também passa a exigir Julia 1.10 (a LTS atual) ou posterior.

A base é a mesma do notebook da 0.7, e cada seção traz a fórmula do que está sendo calculado. A
[página Methods](https://dantebertuzzi.github.io/MissingPatterns.jl/stable/methods/) da documentação tem todas elas, e a
[página References](https://dantebertuzzi.github.io/MissingPatterns.jl/stable/references/) as fontes.

**No Colab, escolha antes o ambiente Julia:** *Ambiente de execução ▸ Alterar o tipo de ambiente ▸ Julia*.

In [ ]:
using Pkg
Pkg.add(["MissingPatterns", "DataFrames"])
Pkg.status("MissingPatterns")   # precisa da 0.8 ou posterior

In [2]:
using MissingPatterns, DataFrames, Dates, Random

## Os dados

Uma tabela simulada de 1.500 notificações, no formato das bases do SUS, com quatro mecanismos de
ausência que não aparecem na tabela:

- **`renda`** é recusada com mais frequência no **Sudeste** e por quem tem **mais de 65 anos**;
- **`glicemia`** e **`peso`** vêm da mesma consulta, que deixou de acontecer com mais frequência
  entre **março de 2020 e dezembro de 2021**;
- **`escolaridade`** traz o "ignorado" codificado como `9`, e **`raca_cor`** traz `"Ignorado"` e
  campos em branco.

In [3]:
Random.seed!(20261011)
n = 1500

regiao      = rand(["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"], n)
data        = Date(2019, 1, 1) .+ Day.(rand(0:2_190, n))
idade       = rand(18:90, n)
escolaridade = [rand() < 0.12 ? 9 : rand(1:5) for _ in 1:n]     # 9 = ignorado (padrão DATASUS)
raca_cor    = [rand() < 0.08 ? "Ignorado" : rand() < 0.03 ? "" :
               rand(["Branca", "Preta", "Parda", "Amarela", "Indígena"]) for _ in 1:n]

renda     = Vector{Union{Missing,Int}}(round.(Int, 1200 .+ 90 .* (idade .- 18) .+ 700 .* randn(n)))
glicemia  = Vector{Union{Missing,Float64}}(round.(95 .+ 15 .* randn(n); digits = 1))
peso      = Vector{Union{Missing,Float64}}(round.(70 .+ 13 .* randn(n); digits = 1))

for i in 1:n
    # recusa em declarar renda: maior no Sudeste e entre os mais velhos
    rand() < 0.08 + 0.27 * (regiao[i] == "Sudeste") + 0.25 * (idade[i] > 65) && (renda[i] = missing)

    # consulta não realizada leva a glicemia e, quase sempre, o peso — mais em 2020–2021
    if rand() < 0.20 + 0.40 * (Date(2020, 3, 1) <= data[i] < Date(2022, 1, 1))
        glicemia[i] = missing
        rand() < 0.85 && (peso[i] = missing)
    end
end

df = DataFrame(; id = 1:n, regiao, data, idade, escolaridade, raca_cor, renda, glicemia, peso)
first(df, 6)

Row,id,regiao,data,idade,escolaridade,raca_cor,renda,glicemia,peso
,Int64,String,Date,Int64,Int64,String,Int64?,Float64?,Float64?
1,1,Sul,2019-09-04,80,1,Ignorado,6654,119.4,61.0
2,2,Sul,2022-04-20,41,3,Indígena,missing,80.9,57.7
3,3,Nordeste,2020-07-29,80,1,Branca,missing,missing,missing
4,4,Norte,2022-04-24,57,3,Parda,4916,102.7,79.7
5,5,Norte,2019-06-25,31,4,Branca,2813,97.8,72.2
6,6,Sudeste,2020-09-22,23,1,Amarela,1835,missing,107.9


## 1. O teste MCAR impresso — e o que ficou de fora

`missingmcar(tbl)` continua devolvendo um `NamedTuple`. Recebendo antes um `IO`, imprime o resultado
com uma leitura. Na tabela inteira, sem `columns`:

In [4]:
missingmcar(stdout, df; color = :never)

 Little's MCAR test on 6 columns (id, idade, escolaridade, renda, glicemia, peso)
 skipped (not real-valued): regiao, data, raca_cor
 d² = 113.2   df = 21   p = 1.25e-14
 ▲ evidence against MCAR at α = 0.05 — the patterns differ systematically, so complete-case analysis is likely biased
 1500 rows used ┊ 6 patterns ┊ EM converged in 19 iterations


A segunda linha é nova. O teste de Little é definido para colunas contínuas; sem `columns`, ele usa
todas as de tipo real e, até a 0.7, descartava as outras sem avisar. Agora elas vêm no campo
`skipped`, e a forma impressa as lista: região, data e raça/cor ficaram de fora.

Vale ler também a primeira linha: o que *entrou* importa tanto quanto o que saiu. `id` é o número da
linha, não uma medida, e `escolaridade` entra com o `9` como se fosse uma categoria acima do
superior. O teste roda, mas o p-valor diz pouco. Escolha as colunas, e deixe `missingsentinels`
transformar os códigos em ausência:

In [5]:
isna = sentinelisna(missingsentinels(df))
missingmcar(stdout, df; isna, columns = [:idade, :renda, :glicemia, :peso], color = :never)

 Little's MCAR test on 4 columns (idade, renda, glicemia, peso)
 d² = 105.8   df = 11   p = 1.25e-17
 ▲ evidence against MCAR at α = 0.05 — the patterns differ systematically, so complete-case analysis is likely biased
 1500 rows used ┊ 6 patterns ┊ EM converged in 20 iterations


## 2. p-valores ajustados — `padj`

`missingpredictors` faz um teste por coluna. Com $m$ testes ao nível $\alpha$, o número esperado de
falsos positivos entre as colunas sem relação chega a $m\alpha$. O notebook da 0.7 resolvia isso à
mão, com Bonferroni (Dunn, 1961). A 0.8 traz o ajuste pronto: com os $m$ p-valores ordenados,
$p_{(1)} \le \dots \le p_{(m)}$, o método de Holm (1979), o padrão, dá

$$
\tilde p_{(i)} = \max_{j \le i}\ \min\bigl(1,\ (m - j + 1)\,p_{(j)}\bigr).
$$

Ele mantém abaixo de $\alpha$ a chance de *qualquer* falso positivo, como Bonferroni, mas nunca
rejeita menos: o menor p-valor é multiplicado por $m$, o segundo por $m - 1$, e assim por diante.
Na forma impressa, as barras das colunas com $\tilde p < \alpha$ vêm cheias:

In [6]:
missingpredictors(stdout, df, :renda; isna, color = :never)

 Missingness of renda: 348 of 1500 rows (23.20%)
 column        test             p      p adj  effect              detail
 regiao        χ²        2.35e-24   1.88e-23    0.28  ███         Sudeste: 37.9% of rows where missing vs 12.4% where present
 idade         welch t   4.27e-23   2.99e-22   +0.63  ██████      mean 63.99 where missing vs 51.29 where present
 escolaridade  welch t      0.001      0.006   +0.21  ██          mean 3.228 where missing vs 2.927 where present
 data          welch t      0.179      0.897   +0.08  ░           mean 2022-01-24 where missing vs 2021-12-02 where present
 id            welch t      0.324      1.000   -0.06  ░           mean 730.9 where missing vs 756.4 where present
 raca_cor      χ²           0.595      1.000    0.05  ░           Parda: 21.8% of rows where missing vs 18.1% where present
 peso          welch t      0.680      1.000   +0.03  ░           mean 70.48 where missing vs 70.1 where present
 glicemia      welch t      0.900      1.000   -0

Região e idade, os dois mecanismos reais, sobrevivem ao ajuste com folga. E a **escolaridade**
também: $p \approx 0{,}001$ vira $\tilde p \approx 0{,}006$, ainda abaixo de 5%. Nada na construção
dos dados liga escolaridade à recusa — é acaso, e o ajuste não o remove.

Isso não é defeito do método. Holm garante que a chance de *algum* falso positivo fique abaixo de
5%; não garante que ele nunca aconteça, e aqui aconteceu. O que denuncia o acaso é o efeito:
$d \approx 0{,}21$, pequeno pela convenção de Cohen, contra $0{,}63$ da idade. O p-valor ajustado
diz quantas colunas merecem atenção; o efeito diz quanto cada uma importa.

`adjust = :bh` usa Benjamini e Hochberg (1995),

$$
\tilde p_{(i)} = \min_{j \ge i}\ \min\Bigl(1,\ \frac{m}{j}\,p_{(j)}\Bigr),
$$

que controla a *proporção* esperada de falsos positivos entre as colunas declaradas significativas
— mais poderoso quando há muitas colunas e vários efeitos reais. Aqui ele concorda com Holm:

In [7]:
missingpredictors(stdout, df, :renda; isna, adjust = :bh, color = :never)

 Missingness of renda: 348 of 1500 rows (23.20%)
 column        test             p      p adj  effect              detail
 regiao        χ²        2.35e-24   1.88e-23    0.28  ███         Sudeste: 37.9% of rows where missing vs 12.4% where present
 idade         welch t   4.27e-23   1.71e-22   +0.63  ██████      mean 63.99 where missing vs 51.29 where present
 escolaridade  welch t      0.001      0.003   +0.21  ██          mean 3.228 where missing vs 2.927 where present
 data          welch t      0.179      0.359   +0.08  ░           mean 2022-01-24 where missing vs 2021-12-02 where present
 id            welch t      0.324      0.519   -0.06  ░           mean 730.9 where missing vs 756.4 where present
 raca_cor      χ²           0.595      0.777    0.05  ░           Parda: 21.8% of rows where missing vs 18.1% where present
 peso          welch t      0.680      0.777   +0.03  ░           mean 70.48 where missing vs 70.1 where present
 glicemia      welch t      0.900      0.900   -0

Onde o ajuste resolve é na glicemia. O notebook da 0.7 apontava a região com $p \approx 0{,}045$ —
passando de raspão nos 5%, sem mecanismo por trás:

In [8]:
missingpredictors(stdout, df, :glicemia; isna, color = :never)

 Missingness of glicemia: 493 of 1500 rows (32.87%)
 column        test             p      p adj  effect              detail
 data          welch t   7.25e-10   5.80e-09   -0.33  ███         mean 2021-08-02 where missing vs 2022-02-18 where present
 regiao        χ²           0.045      0.314    0.08  ░           Centro-Oeste: 25.6% of rows where missing vs 19.8% where present
 peso          welch t      0.381      1.000   +0.10  ░           mean 71.45 where missing vs 70.09 where present
 renda         welch t      0.489      1.000   +0.04  ░           mean 4296 where missing vs 4210 where present
 raca_cor      χ²           0.504      1.000    0.05  ░           Branca: 18.5% of rows where missing vs 16.7% where present
 escolaridade  welch t      0.912      1.000   +0.01  ░           mean 3.005 where missing vs 2.995 where present
 id            welch t      0.945      1.000   -0.00  ░           mean 749.4 where missing vs 751 where present
 idade         welch t      0.958      1.00

Com Holm, a região vai a $\tilde p \approx 0{,}31$ e deixa de ser significativa; só a data, o
mecanismo real, fica. Na forma de dados, `padj` vem entre `pvalue` e `effect`:

In [9]:
select(DataFrame(missingpredictors(df, :glicemia; isna)), :column, :pvalue, :padj, :effect)

Row,column,pvalue,padj,effect
,Symbol,Float64,Float64,Float64
1,data,7.2491e-10,5.79928e-9,-0.329088
2,regiao,0.0448478,0.313934,0.0806244
3,peso,0.38053,1.0,0.103004
4,renda,0.489269,1.0,0.0429587
5,raca_cor,0.503701,1.0,0.0536936
6,escolaridade,0.911727,1.0,0.00653119
7,id,0.945188,1.0,-0.00377099
8,idade,0.957764,1.0,-0.00289657


## 3. Buracos isolados ou blocos — `missinggaps`

Duas colunas com 10% de ausência podem pedir tratamentos opostos. Uma perdeu um valor aqui e ali, que
interpolação ou qualquer imputação resolve; a outra perdeu um trecho contínuo, sem nada observado
por perto para imputar. `missinggaps` mede as **sequências** de linhas ausentes consecutivas.

O caso mais claro é uma série diária. Abaixo, os casos notificados por dia numa unidade perdem um dia
aqui e ali e depois três semanas em agosto, quando o sistema de notificação saiu do ar; os óbitos
perdem só dias espalhados. As duas colunas ficam com o mesmo número de dias ausentes, e o resumo
por coluna mal as distingue:

In [10]:
Random.seed!(3)
dias   = Date(2023, 1, 1):Day(1):Date(2023, 12, 31)
casos  = Vector{Union{Missing,Int}}(rand(0:12, length(dias)))
casos[rand(1:length(dias), 15)] .= missing                          # um dia perdido aqui e ali
casos[Date(2023, 8, 7) .<= dias .<= Date(2023, 8, 27)] .= missing   # ...e três semanas fora do ar
obitos = Vector{Union{Missing,Int}}(rand(0:2, length(dias)))
obitos[rand(1:length(dias), 36)] .= missing                         # só dias espalhados
serie = DataFrame(; dia = collect(dias), casos, obitos)
missingsummary(serie; sortby = :none, color = :never)

 column  type     missing        %  distribution
 dia     Date           0    0.00%                      
 casos   Int64         35    9.59%  ▁▂▁▁▁    ▁ ▅▆▁ ▁  ▁ 
 obitos  Int64         35    9.59%  ▃ ▁▁▂▂▁ ▁▁ ▁ ▂▁▁▂▁▂ 
 70 missing of 1095 cells (6.39%) across 3 columns ┊ bins of 19 rows


In [11]:
missinggaps(serie; by = :dia, color = :never)

 column   missing    gaps  longest  dia                        mean  │ 1   2   3–5 6–10 11–30 31–100 101+
 casos         35      15       21  2023-08-07 … 2023-08-27     2.3  │ ▄▄▄              ▅▅▅▅▅            
 obitos        35      31        3  2023-11-12 … 2023-11-14     1.1  │ ▇▇▇ ▁▁▁ ▁▁▁                       
 46 gaps across 2 columns (42 isolated) ┊ bars: share of each column's missing values lying in a gap of that length ┊ rows in order of dia


Agora elas se separam. Cada barra é a fração dos valores ausentes da coluna que está numa sequência
daquele tamanho — as barras de uma coluna somam todos os seus ausentes —, e os `casos` têm a maior
parte num único buraco de 21 dias, enquanto todas as sequências dos `obitos` são curtas. `by = :dia`
ordena as linhas pela data antes, para que "consecutivo" queira dizer consecutivo no tempo, e o maior
buraco é mostrado em datas.

### Comparando com o acaso

Sequências curtas acontecem por acaso. Se cada linha faltasse de forma independente com
probabilidade $q$, o comprimento $L$ de uma sequência seria geométrico,

$$
P(L = \ell) = q^{\ell - 1}(1 - q), \qquad E[L] = \frac{1}{1 - q},
$$

e o `mean_length` deveria ficar perto de $1/(1-q)$; uma coluna muito acima disso tem blocos. Os
óbitos, com $q \approx 0{,}096$, deveriam ter sequências de $1/(1 - 0{,}096) \approx 1{,}1$ dia em
média, e têm. Os 21 dias seguidos dos casos, com a mesma taxa, o acaso não produz.

Na base de notificações, ordenada pela data:

In [12]:
missinggaps(df; isna, by = :data, color = :never)

 column         missing    gaps  longest  data                       mean  │ 1   2   3–5 6–10 11–30 31–100 101+
 glicemia           493     270        9  2020-10-29 … 2020-11-17     1.8  │ ▃▃▃ ▂▂▂ ▃▃▃ ▂▂▂▂                  
 peso               416     254        7  2020-07-22 … 2020-08-02     1.6  │ ▄▄▄ ▂▂▂ ▃▃▃ ▁▁▁▁                  
 renda              348     257        5  2022-04-17 … 2022-04-21     1.4  │ ▅▅▅ ▃▃▃ ▂▂▂                       
 escolaridade       190     165        4  2020-03-16 … 2020-03-24     1.2  │ ▇▇▇ ▂▂▂ ▁▁▁                       
 raca_cor           158     140        3  2024-07-10 … 2024-07-13     1.1  │ ▇▇▇ ▂▂▂ ▁▁▁                       
 id                   0       0        0  —                             —  │                                   
 regiao               0       0        0  —                             —  │                                   
 idade                0       0        0  —                             —  │                            

A glicemia e o peso têm as sequências mais longas. É um bloco — a pandemia —, ou só uma taxa maior?
A consulta deixava de acontecer com probabilidade 0,60 na pandemia e 0,20 fora dela.
`missinggapstats` devolve os mesmos números como dados, então dá para comparar cada período com a
expectativa geométrica:

In [13]:
s = sort(df, :data)
lacunas(linhas) = only(filter(r -> r.column == :glicemia, missinggapstats(s[linhas, :]; isna)))
pandemia = Date(2020, 3, 1) .<= s.data .< Date(2022, 1, 1)

DataFrame(map([("pandemia", pandemia), ("fora dela", .!pandemia)]) do (periodo, linhas)
    g = lacunas(linhas)
    q = g.nmissing / count(linhas)
    (; periodo, q = round(q; digits = 3), media = round(g.mean_length; digits = 2),
       esperada = round(1 / (1 - q); digits = 2), maior = g.longest)
end)

Row,periodo,q,media,esperada,maior
,String,Float64,Float64,Float64,Int64
1,pandemia,0.625,2.75,2.67,9
2,fora dela,0.208,1.29,1.26,5


Muito perto do esperado nos dois períodos. As sequências são mais longas na pandemia só porque mais
consultas deixaram de acontecer — que é como a base foi construída: cada consulta falhou por conta
própria, nunca um trecho inteiro de uma vez. Para a imputação isso importa: mesmo na pandemia, o maior
buraco tem 9 notificações seguidas, e toda glicemia ausente tem valores observados a poucas
notificações de distância.

`missinggaps` aceita `breaks` para mudar as faixas de tamanho (`[1, 2, 8, 31]` dá 1, 2–7, 8–30 e
31+) e `sortby` para ordenar as colunas.

## Referências

- Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics*, 6(2), 65–70. <https://www.jstor.org/stable/4615733>
- Benjamini, Y., & Hochberg, Y. (1995). Controlling the false discovery rate: a practical and powerful approach to multiple testing. *Journal of the Royal Statistical Society: Series B*, 57(1), 289–300. <https://doi.org/10.1111/j.2517-6161.1995.tb02031.x>
- Dunn, O. J. (1961). Multiple comparisons among means. *Journal of the American Statistical Association*, 56(293), 52–64. <https://doi.org/10.1080/01621459.1961.10482090>
- Little, R. J. A. (1988). A test of missing completely at random for multivariate data with missing values. *Journal of the American Statistical Association*, 83(404), 1198–1202. <https://doi.org/10.1080/01621459.1988.10478722>
- Welch, B. L. (1947). The generalization of 'Student's' problem when several different population variances are involved. *Biometrika*, 34(1–2), 28–35. <https://doi.org/10.1093/biomet/34.1-2.28>
- Rubin, D. B. (1976). Inference and missing data. *Biometrika*, 63(3), 581–592. <https://doi.org/10.1093/biomet/63.3.581>
- Cohen, J. (1988). *Statistical Power Analysis for the Behavioral Sciences* (2nd ed.). Lawrence Erlbaum; reimpresso em 2013, Routledge. <https://doi.org/10.4324/9780203771587>
- Little, R. J. A., & Rubin, D. B. (2019). *Statistical Analysis with Missing Data* (3rd ed.). Wiley. <https://doi.org/10.1002/9781119482260>

## Para continuar

- [Documentação](https://dantebertuzzi.github.io/MissingPatterns.jl/stable) — todas as funções e argumentos; a página [Methods](https://dantebertuzzi.github.io/MissingPatterns.jl/stable/methods/) traz
  a fórmula de cada número
- [`whats-new-0.8.ipynb`](https://colab.research.google.com/github/dantebertuzzi/MissingPatterns.jl/blob/main/notebooks/whats-new-0.8.ipynb) — as mesmas novidades, em inglês
- [`novidades-0.7.ipynb`](https://colab.research.google.com/github/dantebertuzzi/MissingPatterns.jl/blob/main/notebooks/novidades-0.7.ipynb) — os testes em que estas se apoiam
- [`getting-started.ipynb`](https://colab.research.google.com/github/dantebertuzzi/MissingPatterns.jl/blob/main/notebooks/getting-started.ipynb) — o tour completo do pacote